<a href="https://colab.research.google.com/github/gonmpa/Laboratorio_reglas_de_asociacion_MD/blob/main/Actividad_2_6_Reglas_asociaci%C3%B3n_Gonzalo_Pino.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad 2.6 — Laboratorio: Implementación de Reglas de Asociación
**Asignatura:** IEI097 Minería de Datos | **Semana:** 08  
**Algoritmo:** Apriori con `mlxtend`  
**Dataset:** [Bakery Sales Dataset](https://www.kaggle.com/datasets/akashdeepkuila/bakery) (`akashdeepkuila/bakery`)  
**Estudiante:** Gonzalo Pino

In [ ]:
# Instalación de librerías necesarias
!pip install -q mlxtend kagglehub

import os
import pandas as pd
from google.colab import userdata
import kagglehub
from mlxtend.frequent_patterns import apriori, association_rules

# Configuración segura desde Colab Secrets
try:
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("✅ Credenciales de Kaggle cargadas correctamente desde Colab Secrets.")
except Exception as e:
    print("⚠️ Error al cargar credenciales. Verifica que el interruptor 'Notebook access' esté activo en Colab Secrets.", e)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

✅ Credenciales de Kaggle cargadas correctamente desde Colab Secrets.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## Fase 1: Carga y Descripción del Dataset

* **Contexto de negocio:** Registro de ventas transaccionales de la panadería/cafetería *The Bread Basket* en Edimburgo.
* **Definición de Transacción:** Una compra o boleta única efectuada por un cliente, representada por un identificador numérico exclusivo (`TransactionNo`).
* **Definición de Ítem:** Cada producto individual comprado dentro de esa transacción (`Items`).

## Fase 2: Preparación y Codificación One-Hot

**¿Por qué Apriori necesita este formato?**  
El algoritmo Apriori no trabaja con filas duplicadas por transacción. Requiere una matriz booleana binaria donde cada fila sea una transacción única (`TransactionNo`) y cada columna sea un producto (`Items`), indicando con `True` o `False` si el producto estuvo presente en esa compra.

In [ ]:
# 0. Limpiar espacios en los nombres de las columnas
df.columns = df.columns.str.strip()
print("Columnas encontradas en el dataset:", df.columns.tolist())

# Identificar automáticamente la columna de transacción y la de producto
col_trans = [c for c in df.columns if 'trans' in c.lower() or 'no' in c.lower() or 'id' in c.lower()][0]
col_item = [c for c in df.columns if 'item' in c.lower() or 'prod' in c.lower()][0]

print(f"Columna de transacción usada: '{col_trans}'")
print(f"Columna de ítem usada: '{col_item}'")

# 1. Limpieza de valores nulos y espacios en blanco
df = df.dropna(subset=[col_trans, col_item])
df[col_item] = df[col_item].astype(str).str.strip()

# 2. Filtrar filas que no representan productos reales
df = df[~df[col_item].isin(['NONE', 'Adjustment', 'nan', ''])]

# 3. Agrupación para crear la matriz transaccional
basket = df.groupby([col_trans, col_item])[col_item].count().unstack().fillna(0)

# 4. Conversión a matriz binaria (booleana True / False)
basket_encoded = basket > 0

print(f"✅ Matriz binaria generada con éxito: {basket_encoded.shape[0]} transacciones x {basket_encoded.shape[1]} productos.")
basket_encoded.head()

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Columnas encontradas en el dataset: ['Transaction', 'Item', 'date_time', 'period_day', 'weekday_weekend']
Columna de transacción usada: 'Transaction'
Columna de ítem usada: 'Item'
✅ Matriz binaria generada con éxito: 9464 transacciones x 93 productos.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Item,Afternoon with the baker,Alfajores,Argentina Night,Art Tray,Bacon,Baguette,Bakewell,Bare Popcorn,Basket,Bowl Nic Pitt,...,The BART,The Nomad,Tiffin,Toast,Truffles,Tshirt,Valentine's card,Vegan Feast,Vegan mincepie,Victorian Sponge
Transaction,,,,,,,,,,,,,,,,,,,,,
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
5,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## Fase 3: Itemsets Frecuentes

Analizamos la frecuencia de los productos más vendidos para definir nuestro umbral de soporte mínimo (`min_support`) utilizando la librería [`mlxtend`](https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/).

In [ ]:
# Ver el porcentaje de ventas de los 10 productos más populares
top_items = (basket_encoded.sum() / len(basket_encoded)).sort_values(ascending=False).head(10)
print("Top 10 Productos con Mayor Soporte:")
print(top_items.apply(lambda x: f"{x:.4f}"))

# Extracción de conjuntos frecuentes con un soporte mínimo de 0.02 (2% de transacciones)
# Justificación: Un 2% exige que el par de productos aparezca en al menos ~190 compras reales.
frequent_itemsets = apriori(basket_encoded, min_support=0.02, use_colnames=True)
print(f"\nItemsets frecuentes encontrados: {len(frequent_itemsets)}")
frequent_itemsets.sort_values(by="support", ascending=False).head(10)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Top 10 Productos con Mayor Soporte:
Item
Coffee           0.4784
Bread            0.3272
Tea              0.1426
Cake             0.1039
Pastry           0.0861
Sandwich         0.0719
Medialuna        0.0618
Hot chocolate    0.0583
Cookies          0.0544
Brownie          0.0400
dtype: object

Itemsets frecuentes encontrados: 33


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,support,itemsets
4,0.478445,(Coffee)
1,0.327240,(Bread)
16,0.142646,(Tea)
3,0.103867,(Cake)
20,0.090025,"(Coffee, Bread)"
11,0.086116,(Pastry)
12,0.071851,(Sandwich)
9,0.061813,(Medialuna)
7,0.058326,(Hot chocolate)
23,0.054734,"(Cake, Coffee)"


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## Fase 4: Generación de Reglas y Experimento de Umbrales

**Experimento obligatorio:** Comparamos al menos tres combinaciones de umbrales de soporte y confianza para evaluar la sensibilidad del modelo.

In [ ]:
# Evaluación de 3 configuraciones de umbrales
configuraciones = [
    {"soporte": 0.03, "confianza": 0.40},
    {"soporte": 0.02, "confianza": 0.30},
    {"soporte": 0.015, "confianza": 0.20}
]

resultados = []
for config in configuraciones:
    itemsets = apriori(basket_encoded, min_support=config["soporte"], use_colnames=True)
    rules = association_rules(itemsets, metric="confidence", min_threshold=config["confianza"])
    resultados.append({
        "Soporte Mínimo": config["soporte"],
        "Confianza Mínima": config["confianza"],
        "Itemsets Frecuentes": len(itemsets),
        "Reglas Generadas": len(rules)
    })

tabla_exp = pd.DataFrame(resultados)
print("=== Tabla Comparativa de Umbrales ===")
display(tabla_exp)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

=== Tabla Comparativa de Umbrales ===


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Soporte Mínimo,Confianza Mínima,Itemsets Frecuentes,Reglas Generadas
0,0.030,0.4,23,4
1,0.020,0.3,33,10
2,0.015,0.2,46,20


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
# Reglas seleccionadas con soporte=0.02 y confianza=0.30 ordenadas por lift
rules_final = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.30)
rules_final = rules_final.sort_values(by="lift", ascending=False)

print(f"Total de reglas seleccionadas: {len(rules_final)}")
rules_final[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Total de reglas seleccionadas: 10


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,antecedents,consequents,support,confidence,lift
9,(Toast),(Coffee),0.023669,0.704403,1.472276
5,(Medialuna),(Coffee),0.035186,0.569231,1.189753
6,(Pastry),(Coffee),0.047549,0.552147,1.154046
4,(Juice),(Coffee),0.020604,0.534247,1.116632
7,(Sandwich),(Coffee),0.038250,0.532353,1.112674
1,(Cake),(Coffee),0.054734,0.526958,1.101399
2,(Cookies),(Coffee),0.028212,0.518447,1.083608
3,(Hot chocolate),(Coffee),0.029586,0.507246,1.060199
0,(Pastry),(Bread),0.029163,0.338650,1.034868
8,(Tea),(Coffee),0.049873,0.349630,0.730763


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## Fase 5: Interpretación de Negocio y Cierre

### 5 Reglas más relevantes en lenguaje de negocio:
1. **{Toast} -> {Coffee} (Confianza: 70.4%, Lift: 1.47):** Siete de cada diez clientes que piden tostadas también compran un café. El consumo de tostadas incrementa un 47% la probabilidad de adquirir café en comparación a una compra aleatoria.
2. **{Medialuna} -> {Coffee} (Confianza: 56.9%, Lift: 1.19):** La compra de medialunas actúa como un fuerte detonante para la venta de bebidas calientes durante la mañana.
3. **{Pastry} -> {Coffee} (Confianza: 55.2%, Lift: 1.15):** Más de la mitad de los compradores de pasteles agregan café a su pedido.
4. **{Sandwich} -> {Coffee} (Confianza: 53.2%, Lift: 1.11):** En la franja del almuerzo, la compra de sándwiches impulsa de forma directa la venta de café.
5. **{Cake} -> {Coffee} (Confianza: 52.7%, Lift: 1.10):** En el horario de la tarde, acompañar una porción de torta con un café representa el patrón de consumo principal.

### Regla descartada y su justificación técnica:
* **Regla a descartar:** `{Tea} -> {Coffee}` (Fila 8 de la tabla final).
* **Justificación:** Aunque presenta una presencia notable en las transacciones, su valor de **Lift es de 0.73 (inferior a 1.0)**. Un lift menor a 1.0 indica una asociación negativa o de sustitución: quien pide té tiene un 27% *menos* de probabilidades de pedir café. Por ende, no es una combinación útil para estrategias de venta cruzada.

### Decisión comercial propuesta:
* **Combo de Desayuno Promocional:** Diseñar el paquete *Combo Mañanero* (Tostadas + Café) con un 10% de descuento para agilizar la toma de pedidos en horas de alto flujo, y reubicar la vitrina de pasteles justo al lado del punto de entrega del café.

### Reflexión final:
El algoritmo Apriori permitió extraer patrones comerciales claros a partir de miles de compras. El parámetro clave para distinguir reglas valiosas fue el *lift*, ya que la confianza por sí sola sobreestima combinaciones con productos masivos como el café. El experimento de umbrales demostró que reducir el soporte por debajo de 0.015 multiplica la cantidad de reglas redundantes y aumenta innecesariamente la carga de procesamiento.